# IPI Benchmark Analysis

This notebook turns the raw benchmark output into the main paper story.

It answers four questions:

1. Which models are most vulnerable overall?
2. Which attack types work best and which get blocked?
3. Which defenses actually help and by how much?
4. Which model-level patterns stand out?

Run the cells in order after `results/results.csv` has been populated.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

ROOT = Path.cwd().resolve()
BENCHMARK_PATH = ROOT / "benchmark.json"
RESULTS_PATH = ROOT / "results" / "results.csv"
if not RESULTS_PATH.exists() and (ROOT / "results.csv").exists():
    RESULTS_PATH = ROOT / "results.csv"
if not RESULTS_PATH.exists() and (ROOT / "results" / "results_final.csv").exists():
    RESULTS_PATH = ROOT / "results" / "results_final.csv"

OUTPUT_DIR = ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "legend.frameon": False,
})

MODE_ORDER = ["none", "prompt_warning", "spotlighting", "input_filter"]
MODEL_ORDER = [
    "llama3.1_8b",
    "llama33_70b",
    "llama4_scout",
    "gpt_oss_120b",
    "deepseek_r1_distill_qwen32b",
    "phi4",
    "mai_ds_r1",
    "jamba",
    "mistral_small",
    "gpt5",
    "grok3_mini",
    "cohere_command_a",
    "llama32_3b",
    "deepseek_r1",
    "glm45_air",
    "qwen3_80b",
    "llama4_maverick",
    "deepseek_v3",
    "nemotron_super",
    "nous_hermes_405b",
    "liquidai_lfm",
    "gemini3_flash",
    "gemma4_31b",
    "gemma4_26b_moe",
    "claude_haiku",
    "claude_sonnet",
    "claude_opus",
]
MODEL_LABELS = {
    "llama3.1_8b": "LLaMA 3.1 8B",
    "llama33_70b": "LLaMA 3.3 70B",
    "llama4_scout": "LLaMA 4 Scout 17B",
    "gpt_oss_120b": "GPT-OSS 120B",
    "deepseek_r1_distill_qwen32b": "DeepSeek R1 Distill Qwen 32B",
    "phi4": "Phi-4 14B",
    "mai_ds_r1": "MAI-DS-R1",
    "jamba": "Jamba 1.5 Large",
    "mistral_small": "Mistral Small 3.1",
    "gpt5": "GPT-5",
    "grok3_mini": "Grok 3 Mini",
    "cohere_command_a": "Cohere Command A",
    "llama32_3b": "LLaMA 3.2 3B",
    "deepseek_r1": "DeepSeek R1-0528",
    "glm45_air": "GLM-4.5 Air",
    "qwen3_80b": "Qwen3-Next-80B-A3B",
    "llama4_maverick": "LLaMA 4 Maverick",
    "deepseek_v3": "DeepSeek V3-0324",
    "nemotron_super": "Nemotron 3 Super",
    "nous_hermes_405b": "Nous Hermes 3 405B",
    "liquidai_lfm": "Liquid LFM-2.5 1.2B",
    "gemini3_flash": "Gemini 3 Flash",
    "gemma4_31b": "Gemma 4 31B",
    "gemma4_26b_moe": "Gemma 4 26B MoE",
    "claude_haiku": "Claude Haiku 4.5",
    "claude_sonnet": "Claude Sonnet 4.6",
    "claude_opus": "Claude Opus 4.8",
}
MODEL_SIZES_B = {
    "liquidai_lfm": 1.2,
    "llama32_3b": 3,
    "llama3.1_8b": 8,
    "phi4": 14,
    "llama4_scout": 17,
    "mistral_small": 24,
    "gemma4_26b_moe": 26,
    "gemma4_31b": 31,
    "deepseek_r1_distill_qwen32b": 32,
    "grok3_mini": 40,
    "qwen3_80b": 80,
    "jamba": 94,
    "cohere_command_a": 111,
    "gpt_oss_120b": 120,
    "nemotron_super": 120,
    "llama4_maverick": 400,
    "nous_hermes_405b": 405,
    "deepseek_v3": 671,
    "deepseek_r1": 671,
    "mai_ds_r1": 671,
    "gpt5": np.nan,
    "glm45_air": np.nan,
    "gemini3_flash": np.nan,
    "claude_haiku": np.nan,
    "claude_sonnet": np.nan,
    "claude_opus": np.nan,
}
PARADIGM_MAP = {
    "llama3.1_8b": "Standard RLHF",
    "llama33_70b": "Standard RLHF",
    "llama32_3b": "Standard RLHF",
    "llama4_scout": "Meta MoE",
    "gpt_oss_120b": "OpenAI MoE",
    "deepseek_r1_distill_qwen32b": "Reasoning Distill",
    "phi4": "Synthetic data",
    "mai_ds_r1": "Reasoning RL",
    "jamba": "SSM-Transformer hybrid",
    "mistral_small": "European independent",
    "grok3_mini": "Social data",
    "gpt5": "Closed frontier",
    "cohere_command_a": "Agentic MoE",
    "deepseek_r1": "Reasoning RL",
    "glm45_air": "Independent CN",
    "qwen3_80b": "Chinese MoE",
    "llama4_maverick": "Meta Flagship MoE",
    "deepseek_v3": "Chinese MoE",
    "nemotron_super": "Mamba-Transformer hybrid",
    "nous_hermes_405b": "Safety-reduced FT",
    "liquidai_lfm": "Liquid NN",
    "gemini3_flash": "Closed frontier",
    "gemma4_31b": "Google Open",
    "gemma4_26b_moe": "Google Open MoE",
    "claude_haiku": "Constitutional AI",
    "claude_sonnet": "Constitutional AI",
    "claude_opus": "Constitutional AI",
}

def save_figure(filename):
    path = OUTPUT_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to {path}")


In [ ]:
required_columns = {
    "test_id",
    "model_name",
    "defense_mode",
    "prompt_sent",
    "response_received",
    "attack_succeeded",
    "detection_reason",
}

df = pd.read_csv(RESULTS_PATH)
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"results.csv is missing columns: {sorted(missing_columns)}")
if df.empty:
    raise ValueError("results.csv has no rows yet. Run the benchmark before this notebook.")

benchmark = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))
benchmark_df = pd.DataFrame(benchmark)
meta_columns = ["id", "category", "attack_goal", "evasion_style", "setup", "correct_behavior"]
meta = benchmark_df[meta_columns].rename(columns={"id": "test_id"})
df = df.merge(meta, on="test_id", how="left")

df["attack_succeeded"] = pd.to_numeric(df["attack_succeeded"], errors="coerce")
if df["attack_succeeded"].isna().any():
    raise ValueError("attack_succeeded contains missing or non-numeric values.")

missing_meta = df[["category", "attack_goal", "evasion_style"]].isna().sum()
if missing_meta.any():
    print("Warning: some rows did not match benchmark metadata:")
    print(missing_meta.to_string())

print(f"Rows: {len(df):,}")
print(f"Models: {df['model_name'].nunique()}")
print(f"Defense modes: {df['defense_mode'].nunique()}")
print(f"Test cases: {df['test_id'].nunique()}")
print(f"Overall attack success rate: {df['attack_succeeded'].mean():.1%}")
display(df.head())

In [ ]:
main_table = (
    df.pivot_table(
        values="attack_succeeded",
        index="defense_mode",
        columns="model_name",
        aggfunc="mean",
    )
    .reindex(MODE_ORDER)
    .reindex(columns=[model for model in MODEL_ORDER if model in df["model_name"].unique()])
    * 100
).round(1)

main_table_display = main_table.rename(columns=MODEL_LABELS)
display(main_table_display.round(1))
main_table_path = OUTPUT_DIR / "table_main_results.csv"
main_table.to_csv(main_table_path)
print(f"Saved {main_table_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))
heatmap_data = main_table.rename(columns=MODEL_LABELS)
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Attack success rate (%)"},
    ax=ax,
)
ax.set_title("Indirect Prompt Injection Attack Success Rate by Model and Defense", pad=16)
ax.set_xlabel("Model")
ax.set_ylabel("Defense mode")
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
save_figure("figure1_heatmap.png")
plt.show()

## 2. Which defenses actually help, and by how much?

This section compares defense modes against the no-defense baseline and saves the summary for the paper.

In [ ]:
defense_summary = (
    df.groupby("defense_mode")["attack_succeeded"]
    .agg(mean="mean", std="std", count="count")
    .reindex(MODE_ORDER)
    .reset_index()
)
defense_summary["mean_pct"] = defense_summary["mean"] * 100
defense_summary["std_pct"] = defense_summary["std"] * 100
baseline_pct = defense_summary.loc[defense_summary["defense_mode"] == "none", "mean_pct"].iloc[0]
defense_summary["absolute_reduction_pp"] = baseline_pct - defense_summary["mean_pct"]
defense_summary["relative_reduction_pct"] = np.where(
    baseline_pct > 0,
    defense_summary["absolute_reduction_pp"] / baseline_pct * 100,
    np.nan,
)

display(
    defense_summary[["defense_mode", "count", "mean_pct", "absolute_reduction_pp", "relative_reduction_pct"]]
    .style.format({"mean_pct": "{:.1f}", "absolute_reduction_pp": "{:.1f}", "relative_reduction_pct": "{:.1f}"})
)
defense_summary.to_csv(OUTPUT_DIR / "table_defense_summary.csv", index=False)

plot_labels = {
    "none": "No defense",
    "prompt_warning": "Prompt warning",
    "spotlighting": "Spotlighting",
    "input_filter": "Input filter",
}
plot_df = defense_summary.set_index("defense_mode").loc[MODE_ORDER].reset_index()
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    range(len(plot_df)),
    plot_df["mean_pct"],
    color=["#b2182b", "#ef8a62", "#67a9cf", "#2166ac"],
    edgecolor="black",
    linewidth=0.8,
    yerr=plot_df["std_pct"],
    capsize=5,
)
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels([plot_labels[mode] for mode in plot_df["defense_mode"]])
ax.set_ylabel("Attack success rate (%)")
ax.set_title("Defense effectiveness against IPI attacks")
ax.set_ylim(0, 100)
for bar, value in zip(bars, plot_df["mean_pct"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2, f"{value:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.tight_layout()
save_figure("figure2_defense_effectiveness.png")
plt.show()

## 3. Which attack types work best and which get blocked?

This section focuses on the no-defense baseline so attack design is not confounded by defense behavior.

In [ ]:
baseline_df = df[df["defense_mode"] == "none"].copy()

evasion_stats = (
    baseline_df.groupby("evasion_style")["attack_succeeded"]
    .agg(mean="mean", count="count")
    .reset_index()
)
evasion_stats["mean_pct"] = evasion_stats["mean"] * 100
evasion_stats = evasion_stats.sort_values("mean_pct", ascending=False)

goal_stats = (
    baseline_df.groupby("attack_goal")["attack_succeeded"]
    .agg(mean="mean", count="count")
    .reset_index()
)
goal_stats["mean_pct"] = goal_stats["mean"] * 100
goal_stats = goal_stats.sort_values("mean_pct", ascending=False)

category_stats = (
    baseline_df.groupby("category")["attack_succeeded"]
    .agg(mean="mean", count="count")
    .reset_index()
)
category_stats["mean_pct"] = category_stats["mean"] * 100
category_stats = category_stats.sort_values("mean_pct", ascending=False)

combo_stats = (
    baseline_df.groupby(["evasion_style", "attack_goal"])["attack_succeeded"]
    .agg(mean="mean", count="count")
    .reset_index()
)
combo_stats["mean_pct"] = combo_stats["mean"] * 100
combo_stats = combo_stats.sort_values("mean_pct", ascending=False)

display(evasion_stats[["evasion_style", "count", "mean_pct"]].style.format({"mean_pct": "{:.1f}"}))
display(goal_stats[["attack_goal", "count", "mean_pct"]].style.format({"mean_pct": "{:.1f}"}))
display(category_stats[["category", "count", "mean_pct"]].style.format({"mean_pct": "{:.1f}"}))
display(combo_stats.head(10)[["evasion_style", "attack_goal", "count", "mean_pct"]].style.format({"mean_pct": "{:.1f}"}))

evasion_stats.to_csv(OUTPUT_DIR / "table_evasion_style_summary.csv", index=False)
goal_stats.to_csv(OUTPUT_DIR / "table_attack_goal_summary.csv", index=False)
category_stats.to_csv(OUTPUT_DIR / "table_category_summary.csv", index=False)
combo_stats.to_csv(OUTPUT_DIR / "table_attack_combo_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
evasion_colors = sns.color_palette("crest", len(evasion_stats))
axes[0].barh(evasion_stats["evasion_style"], evasion_stats["mean_pct"], color=evasion_colors, edgecolor="black", linewidth=0.6)
axes[0].invert_yaxis()
axes[0].set_xlabel("Attack success rate (%)")
axes[0].set_title("Success rate by evasion style")
for i, (_, row) in enumerate(evasion_stats.iterrows()):
    axes[0].text(row["mean_pct"] + 1, i, f"{row['mean_pct']:.1f}%", va="center")

goal_colors = sns.color_palette("Set2", len(goal_stats))
axes[1].barh(goal_stats["attack_goal"], goal_stats["mean_pct"], color=goal_colors, edgecolor="black", linewidth=0.6)
axes[1].invert_yaxis()
axes[1].set_xlabel("Attack success rate (%)")
axes[1].set_title("Success rate by attack goal")
for i, (_, row) in enumerate(goal_stats.iterrows()):
    axes[1].text(row["mean_pct"] + 1, i, f"{row['mean_pct']:.1f}%", va="center")

plt.tight_layout()
save_figure("figure3_attack_breakdown.png")
plt.show()

## 4. What patterns emerge across model families?

These two cells compare model size and training paradigm against baseline vulnerability.

In [ ]:
baseline_by_model = (
    baseline_df.groupby("model_name")["attack_succeeded"]
    .mean()
    .mul(100)
    .reset_index(name="attack_rate_pct")
)

size_df = pd.DataFrame({
    "model_name": list(MODEL_SIZES_B.keys()),
    "size_b": list(MODEL_SIZES_B.values()),
}).merge(baseline_by_model, on="model_name", how="inner").dropna()
size_df = size_df[size_df["size_b"].notna()].copy()
size_df["log10_size_b"] = np.log10(size_df["size_b"])

if len(size_df) < 2:
    raise ValueError("Not enough models with known size metadata to compute correlation.")

r, p = stats.pearsonr(size_df["log10_size_b"], size_df["attack_rate_pct"])
print(f"Pearson correlation using log10(model size): r={r:.3f}, p={p:.4f}")

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(size_df["size_b"], size_df["attack_rate_pct"], s=100, color="#2b8cbe", edgecolor="black", linewidth=0.8)
for _, row in size_df.iterrows():
    label = MODEL_LABELS.get(row["model_name"], row["model_name"])
    ax.annotate(label, (row["size_b"], row["attack_rate_pct"]), textcoords="offset points", xytext=(6, 5), fontsize=8)

x_grid = np.linspace(size_df["size_b"].min(), size_df["size_b"].max(), 200)
slope, intercept = np.polyfit(size_df["log10_size_b"], size_df["attack_rate_pct"], 1)
ax.plot(x_grid, intercept + slope * np.log10(x_grid), color="#d7301f", linestyle="--", linewidth=2, label=f"log10 trend (r={r:.2f}, p={p:.3f})")
ax.set_xscale("log")
ax.set_xlabel("Model size (billions of parameters, log scale)")
ax.set_ylabel("Baseline attack success rate (%)")
ax.set_title("Model size vs baseline IPI vulnerability")
ax.legend()
plt.tight_layout()
size_df.to_csv(OUTPUT_DIR / "table_model_size_vulnerability.csv", index=False)
save_figure("figure4_size_vs_vulnerability.png")
plt.show()

In [ ]:
paradigm_df = baseline_df.copy()
paradigm_df["paradigm"] = paradigm_df["model_name"].map(PARADIGM_MAP).fillna("Unmapped")

paradigm_stats = (
    paradigm_df.groupby("paradigm")["attack_succeeded"]
    .agg(mean="mean", count="count")
    .reset_index()
)
paradigm_stats["mean_pct"] = paradigm_stats["mean"] * 100
paradigm_stats = paradigm_stats.sort_values("mean_pct", ascending=False)

display(paradigm_stats[["paradigm", "count", "mean_pct"]].style.format({"mean_pct": "{:.1f}"}))
paradigm_stats.to_csv(OUTPUT_DIR / "table_paradigm_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    paradigm_stats["paradigm"],
    paradigm_stats["mean_pct"],
    color=sns.color_palette("coolwarm", len(paradigm_stats)),
    edgecolor="black",
    linewidth=0.6,
)
ax.invert_yaxis()
ax.set_xlabel("Attack success rate (%)")
ax.set_title("Baseline IPI vulnerability by training paradigm")
ax.set_xlim(0, 100)
for bar, value in zip(bars, paradigm_stats["mean_pct"]):
    ax.text(value + 1, bar.get_y() + bar.get_height() / 2, f"{value:.1f}%", va="center", fontsize=10)
plt.tight_layout()
save_figure("figure5_paradigm_comparison.png")
plt.show()

## 5. Paper-ready numbers and exports

This final section collects the headline numbers and saves the annotated dataset plus summary tables.

In [ ]:
results_annotated_path = OUTPUT_DIR / "results_annotated.csv"
model_summary_path = OUTPUT_DIR / "model_summary.csv"
paper_numbers_path = OUTPUT_DIR / "paper_numbers.csv"

df.to_csv(results_annotated_path, index=False)

model_summary = (
    baseline_df.groupby("model_name")
    .agg(attack_rate=("attack_succeeded", "mean"), n_tests=("attack_succeeded", "count"))
    .reset_index()
)
model_summary["attack_rate_pct"] = model_summary["attack_rate"] * 100
model_summary = model_summary.sort_values("attack_rate_pct", ascending=False)
model_summary.to_csv(model_summary_path, index=False)

baseline_rate = baseline_df["attack_succeeded"].mean() * 100
most_resistant = model_summary.sort_values("attack_rate_pct").iloc[0]
most_vulnerable = model_summary.sort_values("attack_rate_pct", ascending=False).iloc[0]
best_defense = defense_summary.sort_values("mean_pct").iloc[0]
strongest_combo = combo_stats.iloc[0]

paper_numbers = pd.DataFrame([
    {"metric": "total_rows", "value": len(df)},
    {"metric": "unique_models", "value": df["model_name"].nunique()},
    {"metric": "unique_defense_modes", "value": df["defense_mode"].nunique()},
    {"metric": "unique_test_cases", "value": df["test_id"].nunique()},
    {"metric": "baseline_attack_success_rate_pct", "value": round(baseline_rate, 2)},
    {"metric": "most_resistant_model", "value": most_resistant["model_name"]},
    {"metric": "most_resistant_rate_pct", "value": round(most_resistant["attack_rate_pct"], 2)},
    {"metric": "most_vulnerable_model", "value": most_vulnerable["model_name"]},
    {"metric": "most_vulnerable_rate_pct", "value": round(most_vulnerable["attack_rate_pct"], 2)},
    {"metric": "best_defense", "value": best_defense["defense_mode"]},
    {"metric": "best_defense_rate_pct", "value": round(best_defense["mean_pct"], 2)},
    {"metric": "strongest_attack_evasion_style", "value": strongest_combo["evasion_style"]},
    {"metric": "strongest_attack_goal", "value": strongest_combo["attack_goal"]},
    {"metric": "strongest_attack_rate_pct", "value": round(strongest_combo["mean_pct"], 2)},
])
paper_numbers.to_csv(paper_numbers_path, index=False)

print("Headline numbers")
print("---------------")
print(f"Rows: {len(df):,}")
print(f"Models: {df['model_name'].nunique()}")
print(f"Defense modes: {df['defense_mode'].nunique()}")
print(f"Test cases: {df['test_id'].nunique()}")
print(f"Baseline attack success rate: {baseline_rate:.1f}%")
print(f"Most resistant model: {most_resistant['model_name']} ({most_resistant['attack_rate_pct']:.1f}%)")
print(f"Most vulnerable model: {most_vulnerable['model_name']} ({most_vulnerable['attack_rate_pct']:.1f}%)")
print(f"Best defense: {best_defense['defense_mode']} ({best_defense['mean_pct']:.1f}%)")
print(f"Strongest attack combo: {strongest_combo['evasion_style']} / {strongest_combo['attack_goal']} ({strongest_combo['mean_pct']:.1f}%)")
print()
print(f"Saved: {results_annotated_path}")
print(f"Saved: {model_summary_path}")
print(f"Saved: {paper_numbers_path}")
print("All figures and tables are saved in the results folder.")